In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_validate, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from palmerpenguins import load_penguins

# --- Load and clean data ---
df = load_penguins().dropna(subset=[
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"
]).drop(columns=["year"])

X = df.drop(columns=["species"])
y = df["species"]

num_features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
cat_features = ["island", "sex"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])

# --- define model candidates ---
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "SVM (RBF)": SVC(kernel="rbf", probability=True, random_state=42),
    "k-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
}

# --- Repeated Stratified CV ---
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
scoring = ["accuracy", "f1_macro", "precision_macro", "recall_macro"]

results = []
for name, clf in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", clf)])
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, return_train_score=False)
    results.append({
        "Model": name,
        "Accuracy (mean ± std)": f"{scores['test_accuracy'].mean():.4f} ± {scores['test_accuracy'].std():.4f}",
        "F1 Macro (mean ± std)": f"{scores['test_f1_macro'].mean():.4f} ± {scores['test_f1_macro'].std():.4f}",
        "Accuracy Mean": scores["test_accuracy"].mean(),
        "Accuracy Std": scores["test_accuracy"].std(),
        "F1 Mean": scores["test_f1_macro"].mean(),
    })

results_df = pd.DataFrame(results).sort_values("Accuracy Mean", ascending=False)
print(results_df[["Model", "Accuracy (mean ± std)", "F1 Macro (mean ± std)"]].to_string(index=False))


              Model Accuracy (mean ± std) F1 Macro (mean ± std)
          SVM (RBF)       0.9947 ± 0.0091       0.9935 ± 0.0114
Logistic Regression       0.9933 ± 0.0089       0.9917 ± 0.0110
k-Nearest Neighbors       0.9930 ± 0.0089       0.9914 ± 0.0109
      Random Forest       0.9895 ± 0.0131       0.9871 ± 0.0165
  Gradient Boosting       0.9851 ± 0.0151       0.9825 ± 0.0181
